# init_recording.ipynb — Synchronous Sensor Recording

This notebook **records**. It produces raw data only.

**Output:** `SESSION_DIR/events.jsonl` + `SESSION_DIR/video.mp4`

These files are then processed by `ground_truth_classifier.ipynb`.

**Hardware sources:**
- **Polar H10** (Bluetooth LE) — R-R intervals
- **Camera** (built-in / Continuity) — video for FER
- **ESP32 + AD8232 + SGP40 + SCD4x** (WebSocket on port 81) — plant voltage @ 100Hz and CO2 (ppm) per batch

> **VOC currently disabled:** the SGP40 VOC sensor is broken. All code that
> reads/writes `voc` events is commented out (not deleted) below — search for
> `VOC disabled` to find every spot. Once a replacement sensor is in, just
> uncomment those lines again.

**Mock mode:** Polar and camera always use real hardware. When `USE_MOCK_ESP32 = True`,
plant voltage and CO2 are synthesised directly into the event store (useful when the
ESP32 is not yet on the network).

**Baseline phase:** the first `BASELINE_S` seconds are a resting phase — all sensors
record, but the camera does not write video/frames yet. `phase` events
(`baseline_start` / `baseline_end`) mark the boundary in `events.jsonl`.


## 1  Imports & Configuration

In [13]:
import os
os.environ["OPENCV_AVFOUNDATION_SKIP_AUTH"] = "1"  # macOS camera fix
import asyncio, json, math, struct, threading, time, warnings
from pathlib import Path

import numpy as np

try:
    from bleak import BleakScanner, BleakClient
    BLEAK_OK = True
except ImportError:
    BLEAK_OK = False
    print("bleak not installed — pip install bleak")

try:
    import cv2
    CV2_OK = True
except ImportError:
    CV2_OK = False
    print("opencv not installed — pip install opencv-python")

try:
    import websockets
    WEBSOCKETS_OK = True
except ImportError:
    WEBSOCKETS_OK = False
    print("websockets not installed — pip install websockets")

# ══════════════════════════════════════════════════════════════════
# CONFIGURATION — adjust here
# ══════════════════════════════════════════════════════════════════
import datetime
PROBAND_ID    = "proband_01"            # ← change per session
_ts           = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")
SESSION_ID    = f"{_ts}_{PROBAND_ID}"
SESSION_DIR   = Path("./sessions") / SESSION_ID
DURATION_S    = 600                      # recording length in seconds
BASELINE_S    = 30                       # resting phase at the start (no video)
VIDEO_FPS     = 30                       # camera frame rate

# ESP32 plant + VOC streamer (see plant_sensor_script.ino)
ESP32_IP        = "10.152.182.238"        # ← IP shown on the OLED screen
ESP32_PORT      = 81
ESP32_RATE_HZ   = 100                    # output rate the ESP32 sends at

# Mock switch — when True, plant_voltage + voc are synthesised locally
# (Polar and camera still need real hardware)
USE_MOCK_ESP32  = False

SESSION_DIR.mkdir(parents=True, exist_ok=True)
EVENT_STORE = SESSION_DIR / "events.jsonl"
VIDEO_FILE  = SESSION_DIR / "video.mp4"

print(f"Session folder : {SESSION_DIR.resolve()}")
print(f"Duration       : {DURATION_S}s  |  Baseline: {BASELINE_S}s")
print(f"ESP32          : {'MOCK' if USE_MOCK_ESP32 else f'ws://{ESP32_IP}:{ESP32_PORT}'}")


Session folder : C:\Uni Koeln\COIN_Project\COIN2026_Team-1_Plant-Emotion-Detector\notebooks\sessions\2026-07-06_19-51_proband_01
Duration       : 600s  |  Baseline: 30s
ESP32          : ws://10.152.182.238:81


## 2  Event Store

All sensors write independently into one shared JSONL file. Each event has
exactly three fields: `ts` (Unix ms), `sensor`, `value`.

```
{"ts": 1700000000000, "sensor": "phase",        "value": "baseline_start"}
{"ts": 1700000000123, "sensor": "hr_rr",        "value": 812.3}
{"ts": 1700000000200, "sensor": "camera_frame",  "value": 42}
{"ts": 1700000001000, "sensor": "co2",           "value": 412.0}
# {"ts": 1700000001000, "sensor": "voc",         "value": 95.0}   ← VOC disabled (SGP40 broken)
{"ts": 1700000001100, "sensor": "plant_voltage", "value": 0.823}
{"ts": 1700000060000, "sensor": "phase",        "value": "baseline_end"}
```

No sensor is the master clock. Resampling onto a common grid happens later in
`ground_truth_classifier.ipynb`.


In [14]:
def write_event(ts_ms, sensor, value):
    """Write one event to the store (thread-safe via append)."""
    with open(EVENT_STORE, "a") as f:
        f.write(json.dumps({"ts": int(ts_ms), "sensor": sensor,
                            "value": value}) + "\n")

# Clear the store (new session)
EVENT_STORE.write_text("")
print(f"Event store initialised: {EVENT_STORE}")


Event store initialised: sessions\2026-07-06_19-51_proband_01\events.jsonl


## 3  Sensor Functions

### 3a  Polar H10 — BLE R-R Intervals

In [15]:
HR_CHAR = "00002a37-0000-1000-8000-00805f9b34fb"

def polar_handler(sender, data):
    """BLE callback: decode R-R intervals (Bluetooth SIG 2011).
    R-R is encoded as little-endian uint16 in units of 1/1024 s."""
    flags      = data[0]
    hr_16_bit  = flags & 0x01
    rr_present = flags & 0x10
    offset     = 3 if hr_16_bit else 2
    if flags & 0x08:
        offset += 2                        # skip Energy Expended field
    if rr_present:
        while offset + 1 < len(data):
            rr_raw = struct.unpack_from("<H", data, offset)[0]
            rr_ms  = rr_raw / 1024 * 1000  # 1/1024 s → ms
            write_event(time.time() * 1000, "hr_rr", round(rr_ms, 1))
            offset += 2

async def run_polar(polar_ready, t0_future, duration_s):
    """Connect to the Polar H10 and stream R-R intervals.

    Two-phase handshake with the caller so BLE discovery (5-10s) happens
    BEFORE the official session start is declared, instead of eating into
    the recording window:
      1. Scan, connect, and call `start_notify` — `polar_handler` writes every
         R-R value to the event store the instant it arrives, with no gating,
         so nothing is lost once the subscription is active.
      2. Set `polar_ready` so the caller knows it is safe to declare T0_S.
      3. Await `t0_future` to learn the real T0_S (resolved by the caller),
         then stay connected and streaming until `t0_s + duration_s`.

    Reconnects automatically on a mid-session drop: `t0_future` is already
    resolved by then, so a reconnect picks up the same stop time.
    """
    if not BLEAK_OK:
        raise RuntimeError("bleak not installed")

    t_end = None
    while t_end is None or time.time() < t_end:
        print("Scanning for Polar H10 ...")
        devices = await BleakScanner.discover()
        polar   = next((d for d in devices if d.name and "Polar" in d.name), None)
        if polar is None:
            print("No Polar H10 found — retrying in 2s ...")
            await asyncio.sleep(2)
            continue
        print(f"Connected: {polar.name}")
        try:
            async with BleakClient(polar.address) as client:
                await client.start_notify(HR_CHAR, polar_handler)
                if not polar_ready.is_set():
                    polar_ready.set()
                t0_s  = await t0_future
                t_end = t0_s + duration_s
                remaining = t_end - time.time()
                if remaining > 0:
                    await asyncio.sleep(remaining)
                await client.stop_notify(HR_CHAR)
        except Exception as e:
            print(f"Polar disconnected ({e}); reconnecting ...")
            await asyncio.sleep(2)
    print("Polar done.")


### 3b  Camera — Frame Timestamps from System Clock

In [16]:
def run_camera(t0_s, duration_s, baseline_s=0):
    import time

    # Prefer index 1 (Mac camera), fall back to 0 (e.g. iPhone Continuity).
    # If only one camera exists, it becomes index 0 → fallback still works.
    selected_index = None
    for i in [1, 0]:
        cap_test = cv2.VideoCapture(i)
        if cap_test.isOpened():
            ok, frame = cap_test.read()
            cap_test.release()
            if ok and frame is not None and frame.size > 0:
                selected_index = i
                print(f"Using camera index {i}")
                break
            else:
                print(f"Camera {i} opened but returned no frames — skipping")

    if selected_index is None:
        print("No working camera found"); return

    cap = cv2.VideoCapture(selected_index)
    time.sleep(2)                       # macOS warm-up
    for _ in range(10):
        cap.read()                      # discard warm-up frames

    # Read resolution from the camera
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0 or fps > 120:
        fps = VIDEO_FPS
    print(f"Resolution: {w}x{h} @ {fps:.0f}fps")

    # avc1 (H.264) works reliably on macOS; fall back to mp4v
    fourcc = cv2.VideoWriter_fourcc(*"avc1")
    writer = cv2.VideoWriter(str(VIDEO_FILE), fourcc, fps, (w, h))
    if not writer.isOpened():
        print("avc1 failed — trying mp4v")
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(str(VIDEO_FILE), fourcc, fps, (w, h))

    if baseline_s > 0:
        print(f"Baseline: {baseline_s}s rest — camera warm, not recording yet")

    # t0_s anchors baseline/end to the GLOBAL session start (same clock the
    # "phase" events use), not to when camera setup happens to finish —
    # camera selection + warm-up above can itself take several seconds, so
    # using a thread-local start time here would let the actual recording
    # start drift away from the baseline_end boundary in the event stream.
    frame_id = 0
    t_record_start = t0_s + baseline_s
    t_end          = t0_s + duration_s
    while time.time() < t_end:
        ok, frame = cap.read()
        if not ok or frame is None or frame.size == 0:
            time.sleep(0.01); continue
        if time.time() < t_record_start:
            continue                     # baseline: stay warm, don't record
        ts_ms = time.time() * 1000       # per-frame system timestamp
        write_event(ts_ms, "camera_frame", frame_id)
        writer.write(frame)
        frame_id += 1

    cap.release()
    writer.release()
    print(f"Camera done: {frame_id} frames -> {VIDEO_FILE}")


### 3c  Environment Sensors — Plant voltage + CO2 (ESP32 / WebSocket) — VOC disabled (SGP40 broken)

In [17]:
async def esp32_listener(t0_s, duration_s):
    """Connect to ESP32 over WebSocket, write voltage + CO2 events
    from `t0_s` until `t0_s + duration_s`. Waits until `t0_s` if started early.
    (VOC disabled — see comments below.)"""
    if not WEBSOCKETS_OK:
        raise RuntimeError("pip install websockets")
    import websockets
    uri = f"ws://{ESP32_IP}:{ESP32_PORT}"

    # Wait until session start
    while time.time() < t0_s:
        await asyncio.sleep(min(0.1, t0_s - time.time()))

    t_end = t0_s + duration_s
    n_batches = 0
    prev_arrival_ms = None
    while time.time() < t_end:
        try:
            async with websockets.connect(uri, ping_interval=20, ping_timeout=10) as ws:
                print(f"ESP32 connected: {uri}")
                async for message in ws:
                    if time.time() >= t_end:
                        break
                    try:
                        data = json.loads(message)
                    except Exception:
                        continue
                    if data.get("type") != "data":
                        continue
                    voltages = data.get("voltages", [])
                    n = len(voltages)
                    if n == 0:
                        continue
                    arrival_ms = time.time() * 1000

                    # Real sample spacing from observed batch cadence
                    # (falls back to the reported rate for the first batch).
                    if prev_arrival_ms is not None and arrival_ms > prev_arrival_ms:
                        step_ms = (arrival_ms - prev_arrival_ms) / n
                    else:
                        step_ms = 1000.0 / data.get("sampleRate", ESP32_RATE_HZ)
                    prev_arrival_ms = arrival_ms

                    # Back-calculate per-sample timestamps
                    for i, mv in enumerate(voltages):
                        ts_ms = arrival_ms - (n - 1 - i) * step_ms
                        volts = float(mv) / 1000.0       # mV -> V
                        write_event(ts_ms, "plant_voltage", round(volts, 6))

                    # CO2 stamped at batch arrival
                    co2 = data.get("co2")
                    if co2 is not None:
                        write_event(arrival_ms, "co2", round(float(co2), 1))

                    n_batches += 1
        except Exception as e:
            print(f"ESP32 disconnected ({e}); retrying in 2s ...")
            await asyncio.sleep(2)
    print(f"ESP32 listener done — {n_batches} batches received")

# ── Mock: synthesise plant + CO2 directly (no fake WebSocket) ──
# VOC disabled (SGP40 broken) — see comments below.
_rng_env = np.random.default_rng(99)

def run_mock_esp32(t0_s, duration_s):
    """Write synthetic plant_voltage @ 100Hz + co2 @ ~2Hz into the
    event store from `t0_s` until `t0_s + duration_s`. Used when
    USE_MOCK_ESP32 = True (e.g. ESP32 not on network)."""
    # Wait until session start
    while time.time() < t0_s:
        time.sleep(0.01)
    t_end = t0_s + duration_s
    next_env = time.time()
    last_v   = 0.50
    while time.time() < t_end:
        ts_ms = time.time() * 1000
        # plant voltage: slow drift + small noise around 0.5V
        last_v += float(_rng_env.normal(0, 0.001))
        last_v  = float(np.clip(last_v, 0.30, 0.70))
        write_event(ts_ms, "plant_voltage", round(last_v, 6))
        # CO2 roughly every 500ms (matches real ESP32 batch cadence)
        if time.time() >= next_env:
            co2 = 450 + float(_rng_env.normal(0, 15))
            write_event(ts_ms, "co2", round(co2, 1))
            next_env = time.time() + 0.5
        time.sleep(0.01)        # 100Hz

def run_env_sensors(t0_s, duration_s):
    """Dispatcher: real ESP32 listener (plant_voltage + co2) or mock.
    VOC disabled — see comments above."""
    if USE_MOCK_ESP32:
        run_mock_esp32(t0_s, duration_s)
    else:
        # Real ESP32 — async WebSocket in its own event loop
        loop = asyncio.new_event_loop()
        try:
            loop.run_until_complete(esp32_listener(t0_s, duration_s))
        finally:
            loop.close()


## 4  Start Recording

All sensors start at the same time. The system clock (`time.time()`) is the
common time base — no sensor is the master.

Plant voltage and HRV (Polar) record continuously from this global start,
including through the baseline — this is intentional, since the within-session
normalisation in `ground_truth_classifier.ipynb` needs baseline data too. The
camera, by contrast, intentionally starts writing frames only after
`BASELINE_S` seconds; it stays warm during baseline but does not record.

**Flow:**
1. Event store already initialised (Block 2)
2. Connect to Polar H10 **first** — BLE scanning takes 5–10s, so this happens
   before the session start is declared; if Polar doesn't connect within
   `POLAR_CONNECT_TIMEOUT_S`, the session is aborted with an error instead of
   starting without it
3. Camera + environment sensors start in their own threads
4. Polar (already connected) starts streaming for `DURATION_S` seconds from
   the now-declared `T0_S`
5. After `DURATION_S` seconds everything stops automatically
6. Short status report

Afterwards: open `events.jsonl` and `video.mp4` in `SESSION_DIR` →
run `ground_truth_classifier.ipynb`.


In [18]:
if USE_MOCK_ESP32:
    print("=" * 62)
    print("  WARNING: USE_MOCK_ESP32 = True")
    print("  plant_voltage / co2 will be SYNTHETIC, not real sensor")
    print("  data. Set USE_MOCK_ESP32 = False in cell 2 for a real")
    print("  recording session. (VOC disabled — SGP40 broken.)")
    print("=" * 62)
    print()

if not BLEAK_OK:
    raise RuntimeError("pip install bleak  (Polar H10 requires bleak)")
if not CV2_OK:
    raise RuntimeError("pip install opencv-python  (camera requires opencv)")
if not USE_MOCK_ESP32 and not WEBSOCKETS_OK:
    raise RuntimeError("pip install websockets  (real ESP32 needs the websockets client)")

# ── Connect Polar BEFORE declaring the session start ──────────────────
# BLE discovery alone takes 5-10s. Doing it here — instead of after T0_S is
# set — means Polar is already connected and streaming by the time the
# official session start is declared below, so no R-R data is lost to the
# scan. If Polar can't connect in time, abort rather than silently starting
# a session with no HRV data.
POLAR_CONNECT_TIMEOUT_S = 20

polar_ready = asyncio.Event()
t0_future   = asyncio.get_running_loop().create_future()
polar_task  = asyncio.create_task(run_polar(polar_ready, t0_future, DURATION_S))

print("Connecting to Polar H10 ...")
try:
    await asyncio.wait_for(polar_ready.wait(), timeout=POLAR_CONNECT_TIMEOUT_S)
except asyncio.TimeoutError:
    polar_task.cancel()
    try:
        await polar_task
    except asyncio.CancelledError:
        pass
    raise RuntimeError(
        f"Polar H10 did not connect within {POLAR_CONNECT_TIMEOUT_S}s — aborting "
        "the session. Check that the device is powered on, in range, and not "
        "already connected elsewhere, then re-run this cell."
    )
print("Polar connected.")

T0_S  = time.time()
T0_MS = int(T0_S * 1000)
# Save session metadata (important for later synchronisation)
meta = {"session_id": SESSION_ID, "proband_id": PROBAND_ID,
        "t0_ms": T0_MS, "duration_s": DURATION_S, "baseline_s": BASELINE_S,
        "esp32_uri": f"ws://{ESP32_IP}:{ESP32_PORT}" if not USE_MOCK_ESP32 else "mock",
        "recorded_at": datetime.datetime.now().isoformat()}
(SESSION_DIR / "session_meta.json").write_text(json.dumps(meta, indent=2))
print(f"Session start : {T0_MS} ms  ({time.strftime('%H:%M:%S')})")
print(f"Baseline      : first {BASELINE_S}s rest — please sit still, no video")
print(f"Total length  : {DURATION_S}s")
print(f"ESP32         : {'MOCK' if USE_MOCK_ESP32 else f'ws://{ESP32_IP}:{ESP32_PORT}'}")
print()

# Mark the baseline boundary directly in the event stream so
# ground_truth_classifier.ipynb doesn't have to re-derive it from
# session_meta.json.
write_event(T0_MS, "phase", "baseline_start")
if BASELINE_S > 0:
    threading.Timer(
        BASELINE_S,
        lambda: write_event(time.time() * 1000, "phase", "baseline_end")
    ).start()

t0_future.set_result(T0_S)   # release Polar to compute its stop time

errors = []

def _run_camera():
    try: run_camera(T0_S, DURATION_S, BASELINE_S)
    except Exception as e: errors.append(f"Camera: {e}")

def _run_env():
    try: run_env_sensors(T0_S, DURATION_S)
    except Exception as e: errors.append(f"Env (ESP32): {e}")

# Start threads
t_cam = threading.Thread(target=_run_camera, daemon=True)
t_env = threading.Thread(target=_run_env,    daemon=True)
t_cam.start()
t_env.start()

# Polar is already connected and running as a background task — just await
# its completion alongside the other sensors.
await polar_task

# Wait for threads to finish
t_cam.join()
t_env.join()

if errors:
    print("\nErrors during recording:")
    for e in errors: print(f"  {e}")
else:
    print("\nRecording complete — no errors")

# Status report
from collections import Counter

# Robust event loader: accept JSON array, JSONL, or concatenated JSON objects per line.
def _load_events_store(path):
    text = Path(path).read_text()
    if not text.strip():
        return []
    try:
        data = json.loads(text)
        if isinstance(data, list):
            return [d for d in data if isinstance(d, dict)]
        return [data] if isinstance(data, dict) else []
    except json.JSONDecodeError:
        pass

    rows = []
    decoder = json.JSONDecoder()
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
            if isinstance(obj, dict):
                rows.append(obj); continue
            if isinstance(obj, list):
                rows.extend([o for o in obj if isinstance(o, dict)]); continue
            continue
        except json.JSONDecodeError:
            pass
        idx = 0; L = len(line)
        while idx < L:
            try:
                obj, end = decoder.raw_decode(line, idx)
                idx = end
                while idx < L and line[idx].isspace(): idx += 1
                if isinstance(obj, dict):
                    rows.append(obj)
                elif isinstance(obj, list):
                    rows.extend([o for o in obj if isinstance(o, dict)])
            except json.JSONDecodeError:
                break
    return rows

events = _load_events_store(EVENT_STORE)
if not events:
    print(f"\nEvent store: empty or unreadable ({EVENT_STORE})")
else:
    by_sensor = Counter(e.get("sensor") for e in events)
    print(f"\nEvent store: {len(events)} events in {EVENT_STORE.name}")
    for s, n in sorted(by_sensor.items()):
        print(f"  {s:<20} {n:>6}")
print(f"\nNext step: run ground_truth_classifier.ipynb")
print(f"SESSION_DIR = '{SESSION_DIR.resolve()}'")


Connecting to Polar H10 ...
Scanning for Polar H10 ...
Connected: Polar H10 E827092E
Polar connected.
Session start : 1783360274870 ms  (19:51:14)
Baseline      : first 30s rest — please sit still, no video
Total length  : 600s
ESP32         : ws://10.152.182.238:81

ESP32 connected: ws://10.152.182.238:81
Using camera index 0
Resolution: 640x480 @ 30fps
Baseline: 30s rest — camera warm, not recording yet
ESP32 listener done — 1424 batches received
Camera done: 17101 frames -> sessions\2026-07-06_19-51_proband_01\video.mp4
Polar done.

Recording complete — no errors

Event store: 90339 events in events.jsonl
  camera_frame          17076
  co2                    1424
  hr_rr                   711
  phase                     2
  plant_voltage         71126

Next step: run ground_truth_classifier.ipynb
SESSION_DIR = 'C:\Uni Koeln\COIN_Project\COIN2026_Team-1_Plant-Emotion-Detector\notebooks\sessions\2026-07-06_19-51_proband_01'
